# 05 — Three-way input EDA

This notebook compares the same selected eight training views in three representations:

1. **Original:** the raw source photograph selected before foreground removal.
2. **Standard input:** BiRefNet foreground removal followed by notebook 03_A's shared scene crop/canvas.
3. **Reflection-handled input:** exactly the same crop, canvas, mask, split, and view order, with only foreground RGB processed by UnReflectAnything.

Statistics use a temporary common 256×256 analysis grid and never modify model inputs. Original-versus-cropped distributions are descriptive because their framing differs. Standard-versus-reflection pixels are aligned, so their paired change measures are meaningful; they quantify change, not restoration quality (there is no diffuse ground truth).


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm


In [ ]:
from pathlib import Path
from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
if not (DRIVE_MOUNT / "MyDrive").is_dir():
    drive.mount(str(DRIVE_MOUNT))
PROJECT_ROOT = DRIVE_MOUNT / "MyDrive" / "ITU" / "3D" / "Thesis"


In [ ]:
import subprocess, sys
CODE_ROOT = Path("/content/Project_Thesis_code")
REPOSITORY = "https://github.com/katlit/Project_Thesis.git"
BRANCH = "codex/hq200-example-notebook"
if not (CODE_ROOT / "code" / "src").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(CODE_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(CODE_ROOT), "pull", "--ff-only"], check=True)
sys.path.insert(0, str(CODE_ROOT / "code")) if str(CODE_ROOT / "code") not in sys.path else None


In [ ]:
from src.eda_utils import masked_image_statistics, paired_foreground_change
from src.image_preprocessing import VIEW_LABELS

STANDARD_ROOT = PROJECT_ROOT / "data_processed" / "method_inputs"
REF_ROOT = PROJECT_ROOT / "data_processed" / "method_inputs_ref"
STANDARD_MANIFEST = STANDARD_ROOT / "manifest.csv"
REF_MANIFEST = REF_ROOT / "manifest.csv"
EDA_ROOT = PROJECT_ROOT / "splits" / "sparse8"
for path in [STANDARD_MANIFEST, REF_MANIFEST]:
    if not path.is_file(): raise FileNotFoundError(f"Required manifest missing: {path}")

standard = pd.read_csv(STANDARD_MANIFEST).query("method == '3dgs' and split == 'train'").copy()
reflected = pd.read_csv(REF_MANIFEST).query("method == '3dgs' and split == 'train'").copy()
keys = ["dataset", "scene", "source"]
reflected = reflected[keys + ["method_image", "method_mask"]].rename(columns={"method_image": "ref_image", "method_mask": "ref_mask"})
paired = standard.merge(reflected, on=keys, how="inner", validate="one_to_one")
counts = paired.groupby(["dataset", "scene"]).size()
assert counts.eq(8).all(), "Each scene must have the same eight standard and reflection-handled inputs."
print(f"Scenes: {len(counts)} | paired views: {len(paired)}")
display(counts.rename("paired_views").to_frame())


In [ ]:
records = []
for row in tqdm(paired.itertuples(index=False), total=len(paired), desc="Three-way statistics"):
    variants = {
        "original": (row.source, row.output_mask),
        "standard_input": (row.method_image, row.method_mask),
        "reflection_handled": (row.ref_image, row.ref_mask),
    }
    for variant, (image_path, mask_path) in variants.items():
        stats = masked_image_statistics(image_path, mask_path, sample_size=(256, 256))
        records.append({"dataset": row.dataset, "scene": row.scene, "source": row.source,
                        "view_order": row.view_order, "variant": variant, **stats})

eda = pd.DataFrame(records)
EDA_ROOT.mkdir(parents=True, exist_ok=True)
eda.to_csv(EDA_ROOT / "image_statistics_three_way.csv", index=False)
summary = eda.groupby(["dataset", "variant"]).agg(
    scenes=("scene", "nunique"), images=("source", "size"),
    width_mean=("width", "mean"), height_mean=("height", "mean"),
    foreground_fraction_mean=("foreground_fraction", "mean"),
    brightness_mean=("brightness", "mean"), contrast_mean=("contrast", "mean"),
    sharpness_mean=("sharpness_proxy", "mean"),
).round(3)
display(summary)
summary.to_csv(EDA_ROOT / "dataset_statistics_three_way.csv")


In [ ]:
paired_changes = []
for row in paired.itertuples(index=False):
    paired_changes.append({"dataset": row.dataset, "scene": row.scene, "source": row.source,
                           "view_order": row.view_order,
                           **paired_foreground_change(row.method_image, row.ref_image, row.method_mask)})
paired_changes = pd.DataFrame(paired_changes)
paired_changes.to_csv(EDA_ROOT / "reflection_paired_changes.csv", index=False)
display(paired_changes.groupby("dataset")[["ref_change_mae", "ref_change_rmse", "ref_change_psnr"]].agg(["mean", "median"]).round(4))


In [ ]:
metrics = ["foreground_fraction", "brightness", "contrast", "sharpness_proxy"]
order = ["original", "standard_input", "reflection_handled"]
colors = ["tab:gray", "tab:blue", "tab:orange"]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for axis, metric in zip(axes.flat, metrics):
    values = [eda.loc[eda.variant.eq(variant), metric].to_numpy() for variant in order]
    axis.boxplot(values, tick_labels=["original", "standard", "reflection"], showfliers=False)
    for patch_color, x, samples in zip(colors, range(1, 4), values):
        jitter = np.random.default_rng(42 + x).normal(x, .035, len(samples))
        axis.scatter(jitter, samples, s=8, alpha=.25, color=patch_color)
    axis.set_title(metric.replace("_", " ").title())
fig.suptitle("Same selected views: three input representations")
plt.tight_layout(); plt.show()


In [ ]:
def show_scene(dataset, scene_name=None):
    part = paired[paired.dataset.eq(dataset)]
    scene_name = scene_name or sorted(part.scene.unique())[0]
    rows = part[part.scene.eq(scene_name)].sort_values(["view_order", "source"])
    fig, axes = plt.subplots(3, 8, figsize=(24, 9), squeeze=False)
    for column, row in enumerate(rows.itertuples(index=False)):
        paths = [row.source, row.method_image, row.ref_image]
        for r, path in enumerate(paths):
            axes[r, column].imshow(Image.open(path).convert("RGB")); axes[r, column].axis("off")
        axes[0, column].set_title(VIEW_LABELS[column])
    for r, label in enumerate(["Original", "Standard input", "Reflection handled"]):
        axes[r, 0].set_ylabel(label, rotation=0, ha="right", labelpad=70)
    fig.suptitle(f"{dataset} / {scene_name}")
    plt.tight_layout(); plt.show()

for dataset in sorted(paired.dataset.unique()):
    show_scene(dataset)


## Reading the comparison

Foreground fraction changes strongly from original to cropped input because notebook 03_A deliberately removes unused canvas; that is expected. Brightness, contrast, and sharpness are computed inside the matching foreground mask on the same analysis resolution. A large standard-to-reflection MAE means the model changed much of the car, not necessarily that it improved it. Use the image grids to check whether glare was removed while lamps, glazing, logos, edges, and paint identity were preserved. Keep both standard and `_ref` branches for downstream ablations.
